# 02 — QLoRA Fine-Tuning on Google Colab (Multi-Seed)

Fine-tunes Qwen2.5-3B-Instruct with QLoRA for cross-lingual sentiment transfer across seeds [42, 123, 456].

**Strategy:** Up to 10 epochs with early stopping (patience=2 on val loss) for each seed.
Best checkpoint is explicitly reloaded at the end via Unsloth, then pushed to seed-specific Hub repos.

**Run this entire notebook top-to-bottom.**

In [ ]:
# Cell 1: Mount Google Drive
# Drive must be mounted BEFORE pip install (requirements file is on Drive)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/qwen-sentiment'
assert os.path.isdir(PROJECT_ROOT), (
    f'Project not found on Drive at {PROJECT_ROOT}. '
    f'Upload your project files to Google Drive first.'
)
os.chdir(PROJECT_ROOT)
os.makedirs('/content/drive/MyDrive/qwen-sentiment/', exist_ok=True)
print(f'✅ Working directory: {os.getcwd()}')
print(f'✅ Files: {sorted(os.listdir("src/"))}')

In [ ]:
# Cell 2: Install dependencies
%pip install -q -r requirements-colab.txt

In [ ]:
# Cell 3: Verify GPU
import torch
assert torch.cuda.is_available(), (
    'No GPU detected! Change runtime to T4: '
    'Runtime > Change runtime type > T4 GPU'
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB')

In [ ]:
# Cell 4: Load API keys
import os

def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

# Priority: Colab Secrets > env var > .env file
for key in ['WANDB_API_KEY', 'HF_TOKEN']:
    val = _colab_secret(key) or os.environ.get(key)
    if not val:
        # Try .env file
        try:
            from dotenv import load_dotenv
            load_dotenv('.env')
            val = os.environ.get(key)
        except ImportError:
            pass
    if val:
        os.environ[key] = val
        print(f'  {key}: loaded ✅')
    else:
        print(f'  {key}: NOT FOUND ⚠️ (will use interactive login)')

In [ ]:
# Cell 5: Authenticate W&B
import wandb
wandb.login()

In [ ]:
# Cell 6: Authenticate Hugging Face
import os
if os.environ.get('HF_TOKEN'):
    print('HF_TOKEN detected in environment. Using it automatically.')
else:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# Cell 7: Import project modules
import torch

from src.dataset import load_all_splits, prepare_english_train_val
from src.train import (
    load_config, load_model_and_tokenizer,
    setup_wandb, create_trainer, train_with_resume,
    push_to_hub, save_locally, log_class_distribution,
    DataCollatorForCompletionOnlyLM,
)

print(f'PyTorch: {torch.__version__}')
print(f'✅ All project modules imported successfully')

In [ ]:
# Cell 8: Load and validate training config + seed list
config = load_config('configs/training_config.yaml')

SEEDS = config['training'].get('seeds', [42, 123, 456])
assert isinstance(SEEDS, list) and len(SEEDS) > 0, 'training.seeds must be a non-empty list'

output_dir = config['checkpointing']['drive_path']
assert output_dir.startswith('/content/drive/'), (
    f'Checkpoint path must be on Drive. Found: {output_dir}'
)
assert config['training'].get('fp16', True) is True, 'fp16 must be True on T4'
assert config['training'].get('bf16', False) is False, 'bf16 must be False on T4'

print(f'✅ Config loaded. Base checkpoint dir: {output_dir}')
print(f'   Seeds: {SEEDS}')
print(f'   Epochs: {config["training"]["num_train_epochs"]}, '
      f'Early stopping patience: {config["training"]["early_stopping_patience"]}')

In [ ]:
# Cell 9: Multi-seed plan (model load/train happen inside the loop)
print('Configured seeds:', SEEDS)
print('Per-seed drive template:', config['checkpointing'].get('drive_path_template', '/content/drive/MyDrive/qwen-sentiment/seed_{seed}/'))
print('Per-seed Hub repo template:', config.get('hub', {}).get('repo_template', 'Mrigank005/SLM-sentiment-crosslingual-seed-{seed}'))

In [ ]:
# Initial load for data formatting — seed loop will reload per-seed
model, tokenizer = load_model_and_tokenizer(config)
print('✅ Initial tokenizer/model loaded for dataset formatting checks')

In [ ]:
# Cell 10: Load and format data
# Uses the SAME tokenizer from model loading — no separate AutoTokenizer!
data = load_all_splits(['english'])
train_ds, val_ds = prepare_english_train_val(data, tokenizer)

print(f'Train: {len(train_ds)} samples')
print(f'Val: {len(val_ds)} samples')

log_class_distribution(train_ds, 'english_train')
log_class_distribution(val_ds, 'english_validation')

# Preview first 3 formatted samples
print('\nFormatted ChatML samples (first 3):')
for i in range(min(3, len(train_ds))):
    print(f'\n--- Sample {i + 1} ---')
    print(train_ds[i]['formatted_text'])

In [ ]:
# Cell 11: Verify loss masking — comprehensive check
# Ensures only assistant response tokens contribute to loss.
# Catches silent all-masked / missing-template failures.

response_template = '<|im_start|>assistant\n'
test_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

# --- Template token ID alignment check ---
template_ids = tokenizer.encode(response_template, add_special_tokens=False)
print(f'Response template: {repr(response_template)}')
print(f'Template token IDs: {template_ids}')
print(f'Template tokens: {[tokenizer.decode([t]) for t in template_ids]}')

def find_subsequence(seq, subseq):
    for i in range(len(seq) - len(subseq) + 1):
        if seq[i:i + len(subseq)] == subseq:
            return i
    return -1

# Check 3 random samples (not just first) to catch data-dependent issues
import random; random.seed(42)
check_indices = [0] + random.sample(range(1, len(train_ds)), min(2, len(train_ds)-1))

for sample_idx in check_indices:
    sample_text = train_ds[sample_idx]['formatted_text']
    tokens = tokenizer(sample_text, return_tensors='pt')
    input_ids = tokens['input_ids'][0].tolist()

    pos = find_subsequence(input_ids, template_ids)
    if pos == -1:
        raise RuntimeError(
            f'CRITICAL: Response template IDs not found in sample {sample_idx}.\n'
            f'DataCollatorForCompletionOnlyLM will mask ALL tokens → zero loss.\n'
            f'Check format_for_sft() produces valid ChatML.'
        )

    batch = test_collator([{'input_ids': input_ids, 'attention_mask': [1]*len(input_ids)}])
    labels = batch['labels'][0]
    n_masked = (labels == -100).sum().item()
    n_total = labels.numel()
    n_active = n_total - n_masked

    assert n_masked > 0, f'Sample {sample_idx}: No tokens masked — loss masking broken!'
    assert n_active > 0, f'Sample {sample_idx}: All tokens masked — no training signal!'
    print(f'  Sample {sample_idx}: {n_total} tokens, {n_masked} masked ({n_masked/n_total:.0%}), '
          f'{n_active} active ({n_active/n_total:.0%}) — template at pos {pos}')

print('\n✅ Loss masking verified on multiple samples. Training signal is intact.')

In [ ]:
# Cell 12: Verify early stopping configuration
t = config['training']
print(f'Max epochs:              {t["num_train_epochs"]}')
print(f'Evaluation strategy:     {t["evaluation_strategy"]}')
print(f'Save strategy:           {t["save_strategy"]}')
print(f'Load best at end:        {t["load_best_model_at_end"]}')
print(f'Early stopping patience: {t["early_stopping_patience"]}')
print(f'Metric monitored:        {t["metric_for_best_model"]} '
      f'(lower is better: {not t["greater_is_better"]})')
assert t['load_best_model_at_end'] == True, 'load_best_model_at_end must be True'
assert t['evaluation_strategy'] == t['save_strategy'], 'eval and save strategy must match'
print(f'\n✅ Early stopping configured: patience={t["early_stopping_patience"]}')

In [ ]:
# Cell 13: Per-seed training loop (load -> train -> push)
import gc, torch, os, json, wandb

for seed in SEEDS:
    print('\n' + '=' * 80)
    print(f'SEED {seed} — START')
    print('=' * 80)

    # Seed-specific config updates
    config['training']['seed'] = int(seed)
    drive_template = config['checkpointing'].get('drive_path_template', '/content/drive/MyDrive/qwen-sentiment/seed_{seed}/')
    config['checkpointing']['drive_path'] = drive_template.format(seed=seed)
    os.makedirs(config['checkpointing']['drive_path'], exist_ok=True)

    # Ensure a fresh W&B run per seed
    try:
        wandb.finish()
    except Exception:
        pass
    config['wandb']['name'] = f'qwen25-3b-qlora-s{seed}'
    config['wandb']['tags'] = ['qwen2.5', 'qlora', 'sentiment', 'cross-lingual', f'seed-{seed}']

    # Remove persisted run id for a clean per-seed run in setup_wandb()
    run_id_path = os.path.join(config['checkpointing']['drive_path'], 'wandb_run_id.json')
    if os.path.exists(run_id_path):
        os.remove(run_id_path)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')

    model, tokenizer = load_model_and_tokenizer(config)
    run_id = setup_wandb(config)
    trainer = create_trainer(model, tokenizer, train_ds, val_ds, config)

    assert trainer.eval_dataset is not None, 'eval_dataset missing from trainer'
    print(f'✅ Trainer created for seed {seed} | W&B run: {run_id}')

    model, tokenizer = train_with_resume(trainer, config)
    print(f'✅ Training complete for seed {seed}')
    print(f'  Best checkpoint: {trainer.state.best_model_checkpoint}')
    print(f'  Best val loss: {trainer.state.best_metric:.4f}')

    repo_template = config.get('hub', {}).get('repo_template', 'Mrigank005/SLM-sentiment-crosslingual-seed-{seed}')
    repo_id = repo_template.format(seed=seed)
    push_to_hub(model, tokenizer, repo_id)
    print(f'✅ Model pushed to: https://huggingface.co/{repo_id}')

    wandb.finish()

In [ ]:
# Cell 14: Optional plot for the final seed run stored in `trainer`
import matplotlib.pyplot as plt
from pathlib import Path

if 'trainer' in globals():
    log_history = trainer.state.log_history
    val_entries = [e for e in log_history if 'eval_loss' in e]
    if val_entries:
        epochs = [e['epoch'] for e in val_entries]
        val_losses = [e['eval_loss'] for e in val_entries]

        best_loss = trainer.state.best_metric

        plt.figure(figsize=(8, 5))
        plt.plot(epochs, val_losses, marker='o', linewidth=2, label='Validation Loss')
        plt.axhline(y=best_loss, color='green', linestyle='--', linewidth=1.5,
                    label=f'Best loss={best_loss:.4f}')
        plt.xlabel('Epoch')
        plt.ylabel('Validation Loss')
        plt.title('Qwen2.5 3B QLoRA — Validation Loss per Epoch (last seed run)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        save_path = Path('results/training_curves/val_loss_per_epoch_last_seed.png')
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'Best checkpoint: {trainer.state.best_model_checkpoint} | Best val loss: {best_loss:.4f}')
    else:
        print('No evaluation history found in trainer.state.log_history.')
else:
    print('Run Cell 13 first to create `trainer`.')